# 🏨 Hotel Occupancy & Room-Demand Forecasting: Exploratory Data Analysis (EDA)

**Objective**: Analyze historical booking dynamics, lead-time velocity (booking pace), cancellation behavior, seasonality, and Indian festival demand uplift.

### Standout Concept: The Booking Pace (Pickup) Curve
Unlike generic time-series, hotel occupancy is not a black-box daily number. On any prediction date $T$, reservations are already On-The-Books ($OTB$) for future stay dates. What we forecast is the **net pickup** between today and arrival.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Ensure project root is in path
sys.path.insert(0, '..')
from src.config import RAW_BOOKINGS_CSV, ROOM_CATEGORIES

# Load data
df = pd.read_csv(RAW_BOOKINGS_CSV)
df['booking_date'] = pd.to_datetime(df['booking_date'])
df['checkin_date'] = pd.to_datetime(df['checkin_date'])
df['checkout_date'] = pd.to_datetime(df['checkout_date'])
df['cancellation_date'] = pd.to_datetime(df['cancellation_date'])

print(f"Total Booking Records: {len(df):,}")
print(f"Stay Date Range: {df['checkin_date'].min().date()} to {df['checkin_date'].max().date()}")
df.head()

## 1. Summary Statistics by Room Category

In [ ]:
summary = df.groupby('room_category').agg(
    total_bookings=('booking_id', 'count'),
    cancellation_rate=('is_cancelled', lambda x: f"{x.mean()*100:.1f}%"),
    median_lead_time=('lead_time', 'median'),
    mean_adr=('adr', lambda x: f"₹{x.mean():,.2f}")
).reset_index()
summary

## 2. Booking Pace (Pickup) Velocity Curves
Cumulative bookings on the books as the arrival date approaches (from 60 days out down to 0 days).

In [ ]:
horizons = np.arange(0, 61)
pace_records = []
sample_dates = df['checkin_date'].drop_duplicates().sample(n=100, random_state=42)
sample_df = df[df['checkin_date'].isin(sample_dates)]

for cat in ROOM_CATEGORIES.keys():
    cat_df = sample_df[sample_df['room_category'] == cat]
    for h in horizons:
        otb_mask = (cat_df['lead_time'] >= h) & (
            (cat_df['is_cancelled'] == 0) | 
            ((cat_df['checkin_date'] - cat_df['cancellation_date']).dt.days < h)
        )
        avg_otb = otb_mask.sum() / len(sample_dates)
        pace_records.append({'horizon_days_out': h, 'room_category': cat, 'avg_otb': avg_otb})

pace_df = pd.DataFrame(pace_records)

plt.figure(figsize=(9, 5))
sns.lineplot(data=pace_df, x='horizon_days_out', y='avg_otb', hue='room_category', linewidth=2.5)
plt.gca().invert_xaxis()
plt.title('Booking-Pace Curves: On-The-Books Reservations Countdown to Arrival', fontsize=12, weight='bold')
plt.xlabel('Days Before Arrival (60d → 0d Check-in)')
plt.ylabel('Average Rooms On-The-Books')
plt.axvline(14, color='gray', linestyle='--', label='14-Day Horizon')
plt.axvline(7, color='red', linestyle='--', label='7-Day Horizon')
plt.legend()
plt.show()

## 3. Seasonality Patterns: Day-of-Week & Monthly Trends

In [ ]:
realized = df[(df['is_cancelled'] == 0) & (df['is_no_show'] == 0)].copy()
daily_occ = realized.groupby(['checkin_date', 'room_category']).size().reset_index(name='occupied_rooms')
cap_map = {k: v['capacity'] for k, v in ROOM_CATEGORIES.items()}
daily_occ['occupancy_rate'] = (daily_occ['occupied_rooms'] / daily_occ['room_category'].map(cap_map)) * 100
daily_occ['dow'] = daily_occ['checkin_date'].dt.day_name()
daily_occ['month'] = daily_occ['checkin_date'].dt.strftime('%b')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
sns.barplot(data=daily_occ, x='dow', y='occupancy_rate', hue='room_category', order=dow_order, ci=None, ax=axes[0])
axes[0].set_title('Weekly Occupancy Rate (%) by Day of Week', weight='bold')
axes[0].tick_params(axis='x', rotation=30)

month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
sns.barplot(data=daily_occ, x='month', y='occupancy_rate', order=month_order, color='#3470a3', ci=None, ax=axes[1])
axes[1].set_title('Monthly Seasonality (Winter Peak vs Monsoon Trough)', weight='bold')
plt.tight_layout()
plt.show()

## 4. Cancellation Risk by Lead Time Bucket and Channel

In [ ]:
bins = [-1, 7, 14, 30, 60, 200]
labels = ['0-7d (Last Min)', '8-14d (1-2 Wk)', '15-30d (2-4 Wk)', '31-60d (1-2 Mo)', '>60d (Long Lead)']
df['lead_time_bucket'] = pd.cut(df['lead_time'], bins=bins, labels=labels)

lt_canc = df.groupby(['lead_time_bucket', 'market_segment'])['is_cancelled'].mean().reset_index()
lt_canc['cancellation_rate_pct'] = lt_canc['is_cancelled'] * 100.0

plt.figure(figsize=(10, 5))
sns.barplot(data=lt_canc, x='lead_time_bucket', y='cancellation_rate_pct', hue='market_segment', palette='muted')
plt.title('Cancellation Rate (%) Across Lead Time and Booking Channel', weight='bold')
plt.ylabel('Cancellation Rate (%)')
plt.show()

## 5. Indian Festival & Holiday Uplift

In [ ]:
realized['is_festival'] = realized['festival_event'] != 'None'
fest_summary = realized.groupby(['is_festival', 'room_category']).agg(
    mean_adr=('adr', 'mean'),
    total_stays=('num_rooms', 'sum')
).reset_index()
fest_summary['Event Type'] = fest_summary['is_festival'].map({True: 'Festival / Holiday', False: 'Normal Days'})
fest_summary